# Agência Nacional de Mineração

No que tange aos dados espaciais, a _Agência Nacional de Mineração_ (AMN) mantem o portal [SIGMINE](https://geo.anm.gov.br/portal/apps/webappviewer/index.html?id=6a8f5ccc4b6a4c2bba79759aa952d908).


In [1]:
import tempfile
from pathlib import Path

import folium
from folium import plugins

import open_geodata as geo

<br>

---

## Lista Layers


In [2]:
# Instancia classe IBGE
amn = geo.br.amn.ANM()

# Obtem Malhas de 2018
layers = amn.layers

# Resultados
layers.info()
layers.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   nome    6 non-null      object
 1   url     6 non-null      object
dtypes: object(2)
memory usage: 228.0+ bytes


,nome,url
0,Áreas de Bloqueio,https://app.anm.gov.br/dadosabertos/SIGMINE/BL...
1,Processos Minerários Ativos - SP,https://app.anm.gov.br/dadosabertos/SIGMINE/PR...
2,Arrendamentos,https://app.anm.gov.br/dadosabertos/SIGMINE/AR...
3,Áreas de Proteção de Fonte,https://app.anm.gov.br/dadosabertos/SIGMINE/PR...
4,Áreas de Servidão,https://app.anm.gov.br/dadosabertos/SIGMINE/AR...


<br>

---

## Obtem Dados


In [3]:
gdf = amn.get_layers(layer='Processos Minerários Ativos - SP')
gdf.info()
gdf.head(2)

d:\Codes\GitHub\Personal\my_projects\open-geodata\.venv\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 17199 entries, 0 to 17198
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   PROCESSO    17199 non-null  object  
 1   NUMERO      17199 non-null  int32   
 2   ANO         17199 non-null  int32   
 3   AREA_HA     17199 non-null  float64 
 4   ID          17199 non-null  object  
 5   FASE        17199 non-null  object  
 6   ULT_EVENTO  17199 non-null  object  
 7   NOME        17199 non-null  object  
 8   SUBS        17199 non-null  object  
 9   USO         17199 non-null  object  
 10  UF          17199 non-null  object  
 11  DSProcesso  17199 non-null  object  
 12  geometry    17199 non-null  geometry
dtypes: float64(1), geometry(1), int32(2), object(9)
memory usage: 1.6+ MB


,PROCESSO,NUMERO,ANO,AREA_HA,ID,FASE,ULT_EVENTO,NOME,SUBS,USO,UF,DSProcesso,geometry
0,3984/1935,3984,1935,32.34,{B96214CD-DC57-4BFF-AD09-9E60A3B6E01F},CONCESSÃO DE LAVRA,473 - CONC LAV/CUMPRIMENTO EXIGÊNCIA PROTOC EM...,FUMEST - FOMENTO DE URBANIZACAO E MELHORIA DAS...,ÁGUA MINERAL,Demais substâncias,SP,003.984/1935,"POLYGON Z ((-49.17876 -21.1034 0, -49.17804 -2..."
1,2706/1936,2706,1936,1.04,{8772AB4C-8EDD-46B7-B961-6B8240575003},CONCESSÃO DE LAVRA,418 - CONC LAV/RAL ANO BASE APRESENTADO EM 20/...,GUAPIARA MINERACAO INDUSTRIA E COMERCIO LTDA,CALCÁRIO,Demais substâncias,SP,002.706/1936,"POLYGON Z ((-47.55343 -23.64753 0, -47.55308 -..."


<br>

---

## Filtra Dados


In [4]:
# Search
word_search = 'Mineração Righi Ltda Me'

# Select
gdf_interess = gdf.loc[gdf['NOME'].str.contains(word_search, case=False)]
gdf_interess

,PROCESSO,NUMERO,ANO,AREA_HA,ID,FASE,ULT_EVENTO,NOME,SUBS,USO,UF,DSProcesso,geometry
4199,821382/1999,821382,1999,49.98,{0A441C93-457C-441B-8ABF-A8844EB74A25},CONCESSÃO DE LAVRA,436 - CONC LAV/DOCUMENTO DIVERSO PROTOC EM 10/...,Mineração Righi Ltda Me,AREIA,Demais substâncias,SP,821.382/1999,"POLYGON Z ((-47.78647 -22.45755 0, -47.77953 -..."
7374,820417/2005,820417,2005,15.14,{D46C2AE9-B20B-40EE-B1BB-DD2B2CD1FDBA},REQUERIMENTO DE LAVRA,336 - REQ LAV/DOCUMENTO DIVERSO PROTOC EM 26/0...,Mineração Righi Ltda Me,AREIA,Construção civil,SP,820.417/2005,"POLYGON Z ((-47.78384 -22.45755 0, -47.78384 -..."


<br>

---

## Folium


In [5]:
# PopUp
def popup_html(row):
    # Data
    # r01 = row['PROCESSO']
    # r02 = row['NUMERO']
    # r03 = row['ANO']
    r04 = row['AREA_HA']
    # r05 = row['ID']
    r06 = row['FASE']
    r07 = row['ULT_EVENTO']
    r08 = row['NOME']
    r09 = row['SUBS']
    # r10 = row['USO']
    # r11 = row['UF']
    r12 = row['DSProcesso']
    # r13 = row['geometry']

    # Infos
    return f"""
    <div>
    <h5>Processo {r12}</h5>
    <br><b>Nome:</b> {r08}
    <br><b>Substância:</b> {r09}
    <br>--------------------
    <br><b>Área (ha):</b> {r04}
    <br><b>Fase:</b> {r06}
    </div>
    """

<br>

---

### Create Layer


In [6]:
def add_lyr_righi(geodataframe):
    # Input

    gdf = geodataframe.to_crs(epsg=4326)

    # Popup
    gdf['popup'] = gdf.apply(popup_html, axis=1)

    # Layer
    lyr = folium.GeoJson(
        gdf,
        name='Processos Minerários - Righi',
        smooth_factor=1.0,
        style_function=lambda x: {
            'fillColor': '#DC143C',
            'color': '#DC143C',
            'weight': 1,
            'fillOpacity': 0.3,
        },
        highlight_function=lambda x: {
            'weight': 3,
            'fillOpacity': 0.6,
        },
        popup=folium.GeoJsonPopup(
            ['popup'],
            parse_html=False,
            max_width='400',
            show=False,
            labels=False,
            sticky=True,
        ),
        marker=folium.Marker(
            icon=folium.Icon(
                color='lightgray',
                icon_color='#FFFF00',
                # icon='leaf',
            ),
        ),
        tooltip=folium.GeoJsonTooltip(
            fields=['NOME'],
            aliases=['NOME'],
            sticky=True,
            opacity=0.9,
            direction='right',
        ),
        embed=False,
        zoom_on_click=False,
        control=True,
        show=True,
    )
    return lyr

In [7]:
def get_map(input_geojson):
    # Input
    # gdf = gpd.read_file(input_geojson)
    gdf = input_geojson.to_crs(epsg=4326)
    sw = gdf.bounds[['miny', 'minx']].min().values.tolist()
    ne = gdf.bounds[['maxy', 'maxx']].max().values.tolist()
    bounds = [sw, ne]

    # Zoom
    min_zoom = 6
    max_zoom = 21
    padding = 1

    # Create Map
    m = folium.Map(
        min_zoom=min_zoom,
        max_zoom=max_zoom,
        max_bounds=True,
        min_lat=bounds[0][0] * ((100 + padding) / 100),
        min_lon=bounds[0][1] * ((100 + padding) / 100),
        max_lat=bounds[1][0] * ((100 - padding) / 100),
        max_lon=bounds[1][1] * ((100 - padding) / 100),
        tiles=None,
        # zoom_delta=0.1,
        # zoom_start=10,
    )

    # Add Layers
    m.add_child(geo.lyr.base.google_hybrid(min_zoom, max_zoom))
    m.add_child(geo.lyr.base.google_satellite(min_zoom, max_zoom))
    m.add_child(geo.lyr.base.google_terrain(min_zoom, max_zoom))
    m.add_child(geo.lyr.base.google_streets(min_zoom, max_zoom))

    # Monitoramento
    m.add_child(add_lyr_righi(input_geojson))

    # Plugins
    m.fit_bounds(bounds)
    plugins.Fullscreen(
        position='topleft',
        title='Clique para Maximizar',
        title_cancel='Mininizar',
    ).add_to(m)
    folium.LayerControl(
        position='topright',
        collapsed=False,
    ).add_to(m)
    return m

In [8]:
word_search_outfilename = word_search.lower().replace(' ', '_')
word_search_outfilename

'mineração_righi_ltda_me'

In [9]:
with tempfile.TemporaryDirectory() as temp_dir:
    # Path
    temp_dir = Path(temp_dir)

    # Create Maps
    m = get_map(input_geojson=gdf_interess)

    # Save
    m.save(temp_dir / f'map_{word_search_outfilename}.html')

In [10]:
m